In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window
emp_raw_df = spark.table("default.employee_snapshot")
dept_raw_df = spark.table("default.department_master")
def typeCasting(df,column, datatype):
    new_df = df.withColumn(column, df[column].try_cast(datatype))
    return new_df
def standerdize_Column(df,column):
    new_df = df.withColumn(column, F.upper(F.trim(df[column])))
    return new_df
def deduplicate_table(df,partition_col,order_col):
    window = Window.partitionBy(partition_col).orderBy(F.col(order_col).desc())
    dedup_df = df.withColumn('rn', F.row_number().over(window))\
                .filter(F.col("rn") == 1)\
                .drop("rn")
    return dedup_df
# def history_table(df):
#     df.write.mode("append").saveAsTable("workspace.default.emp_audit_for_history")
#     new_df = spark.sql("SELECT * FROM workspace.default.emp_audit_for_history")
#     return new_df
emp_cleaned_sal = typeCasting(emp_raw_df,"salary","double")
emp_cleaned_date = typeCasting(emp_cleaned_sal,"join_date","date")
emp_clean = standerdize_Column(emp_cleaned_date,"dept")
emp_dedup = deduplicate_table(emp_clean, "emp_id", "join_date")
dept_clean = standerdize_Column(dept_raw_df,"dept")
# ================================================================================================
validate_emp = emp_dedup.join(dept_clean, on= "dept", how= "left")
invalid_emp = emp_dedup.join(dept_clean, on = "dept", how = "left_anti")
final_valid = validate_emp\
              .withColumn("record_stat", F.lit("VALID"))\
              .withColumn("processed_ts",F.current_timestamp())
final_invalid = invalid_emp\
              .withColumn("record_stat", F.lit("INVALID"))\
              .withColumn("processed_ts",F.current_timestamp())
# history_table(final_invalid)
# history_table(final_valid)

# spark.sql("""
# ALTER TABLE workspace.default.emp_audit_for_history
# ADD COLUMNS (location STRING)
# """)

final_valid.write \
    .mode("append") \
    .saveAsTable("workspace.default.emp_audit_for_history")
spark.sql("SELECT * FROM workspace.default.emp_audit_for_history").show()
# validate_emp.show()
# invalid_emp.show()
# emp_clean.printSchema()
# emp_clean.show()
# emp_dedup.show()
final_valid.show()
final_invalid.show()



